In [1]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim

import torch_geometric
from torch_geometric.nn import knn_graph
from torch_geometric.loader import DataLoader

import numpy as np
import random
from sklearn.model_selection import train_test_split
import wandb

import gc

from segnn.segnn import SEGNN
from e3nn.o3 import Irreps, spherical_harmonics
from segnn.balanced_irreps import BalancedIrreps, WeightBalancedIrreps
from segnn.instance_norm import InstanceNorm
# from Utility_functions import Graph_datasetV2 as Graph_dataset

# use it for input features similar to hemodynamics paper
from Utility_functions import Graph_dataset_with_equiv_features, inlet_distance_mask

import params

print('DATADIR', params.DATADIR)
print('NSIM', params.NSIM)
print('BATCH_SIZE', params.BATCH_SIZE)

print('python.__version__', sys.version_info)
print('torch.__version__', torch.__version__)
print('torch_geometric.__version__', torch_geometric.__version__)
print('torch.cuda.is_available()', torch.cuda.is_available())


DATADIR ../.data/Dataset_10sims_5G2N
NSIM 10
BATCH_SIZE 1
python.__version__ sys.version_info(major=3, minor=8, micro=12, releaselevel='final', serial=0)
torch.__version__ 1.10.1
torch_geometric.__version__ 2.0.3
torch.cuda.is_available() False


In [2]:
a = Irreps('1o+2e+3x0e')
b = Irreps('4x1o+2o+2x1e')

c = Irreps()

In [3]:
print("PyTorch has version {}".format(torch.__version__))
print("The linked CUDA version is", torch.version.cuda)

dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# dev = torch.device("cpu")
# torch.cuda.init()

print('Running on device:', dev)

PyTorch has version 1.10.1
The linked CUDA version is 11.3
Running on device: cpu


# Reproducibility seeds

In [4]:
seed = 0

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Hyperparameters

In [5]:
class objectview(object):
    def __init__(self, d):
        self.__dict__ = d

for args in [
        {'epochs': 3,
         'batch_size': 4,          
         'input_size' : 8, # <- only for reference
         'edge_lmax' : 1,
         'node_lmax' : 1,
         'hidden_lmax' : 1,
         'num_layers' : 8,
         'task' : 'node',
         'norm' : 'batch',
         'output_size' : 4,
         'hidden_size': 256, 
         'neighbours' : 3, # 10 for moebius data
         'subsample_dataset': 1,
         'opt': 'Adam', 
        #  'scheduler': 'ExponentialLR', 
         'scheduler': None, 
         'learning_rate': 3e-4,
         'device': dev,
         'early_stop' : 5}
    ]:
        args = objectview(args)

# **Dataset and dataloader**

In [6]:
# Dataset, change the root path accordingly
dataset = Graph_dataset_with_equiv_features(root = params.DATADIR)  #, inlet_mask = True)

# 80-10-10 split

# first the dataset is split 80%-20%, with the first portion being the training set

if params.NSIM == 1:
    train_idx = list(range(1))
    val_test_idx = []
elif params.NSIM == 10:  # in this case the test_size cannot be smaller than 2
    train_idx, val_test_idx = train_test_split(range(len(dataset)), test_size = 0.2, shuffle = False)
else:
    train_idx, val_test_idx = train_test_split(range(len(dataset)), test_size = 0.1, shuffle = False)

print(train_idx)
print(val_test_idx)
# the 20% portion is split in half so to have two sets with 10% data each of the original dataset
if params.NSIM == 1:
    val_idx, test_idx = [], []
else:
    val_idx, test_idx = train_test_split(range(len(val_test_idx)), test_size = 0.5, shuffle = False)

# this line can be used to do training on a smaller subset of the training set
# if args.subsample_dataset != 1
train_idx = train_idx[:(len(train_idx))// args.subsample_dataset] 

train_dataset = dataset[train_idx]
val_dataset = dataset[val_idx]

print(len(train_idx), len(val_idx), len(test_idx))
print(len(train_idx) / len(dataset), len(val_idx) / len(dataset), len(test_idx) / len(dataset))

# torch_geometric DataLoaders are used for handling lists of graphs
n_works = 0
t_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, num_workers=n_works)#, persistent_workers=True)
v_loader = DataLoader(val_dataset, batch_size=args.batch_size, shuffle=False, num_workers=n_works)#, persistent_workers=True)

100%|██████████| 10/10 [00:00<?, ?it/s]

[0, 1, 2, 3, 4, 5, 6, 7]
[8, 9]
8 1 1
0.8 0.1 0.1


In [7]:
name = 'SEGNN_model'
project = "local_SEGNN_with_inlet_trials"

In [8]:
for sample in t_loader:
    break

In [9]:
print(sample)
print(sample.node_attr[:])

# checking number of zero entries on node attributes; they are a lot!
print((torch.sum(sample.node_attr,dim=-1)==0).sum())

DataBatch(x=[240, 8], y=[240, 4], pos=[240, 3], node_attr=[240, 4], original_pos_plus_labels=[248, 4], mask=[240], batch=[240], ptr=[5])
tensor([[ 0.0000e+00,  0.0000e+00,  0.0000e+00, -1.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00, -1.0000e+00],
        [ 0.0000e+00,  7.7426e-03,  2.5238e-01, -2.9033e-01],
        [ 0.0000e+00,  7.7426e-03,  2.5238e-01, -2.9033e-01],
        [ 0.0000e+00,  2.4947e-02, -1.8250e-01, -5.8701e-01],
        [ 0.0000e+00,  2.4947e-02, -1.8250e-01, -5.8701e-01],
        [ 0.0000e+00,  8.9209e-02,  8.3930e-02, -1.0942e-01],
        [ 0.0000e+00,  8.9209e-02,  8.3930e-02, -1.0942e-01],
        [ 0.0000e+00, -3.4442e-02,  1.3607e-01, -1.7009e-01],
        [ 0.0000e+00, -3.4442e-02,  1.3607e-01, -1.7009e-01],
        [ 0.0000e+00, -1.3650e-01, -1.0920e-01, -1.9104e-01],
        [ 0.0000e+00, -1.3650e-01, -1.0920e-01, -1.9104e-01],
        [ 0.0000e+00,  1.2426e-01, -7.2288e-02, -3.2599e-01],
        [ 0.0000e+00,  1.2426e-01, -7.2288e-02, -3.2599e-

# **Model**

In [10]:
gc.collect()
torch.cuda.empty_cache()
print()
print(torch.cuda.memory_allocated()*4/(1024**2), "MB")
print(torch.cuda.max_memory_allocated()*4/(1024**2), "MB")
print()
# print(torch.cuda.memory_summary())


0.0 MB
0.0 MB



In [11]:
!nvidia-smi

"nvidia-smi" non � riconosciuto come comando interno o esterno,
 un programma eseguibile o un file batch.


In [12]:
gc.collect()
torch.cuda.empty_cache()

#input_irreps=Irreps('5x0e'), # x: node features [SDF,MIS,3x0e one_hot encodng]


# x: node features for fluid nodes [distance vector to closest inlet node,
#                                   distance vector to closest outlet node,
#                                   SDF (rel_dist to wall)]
# 2x1o + 0e means 2 vectors (odd parity) and one scalar (even parity)
input_irreps = Irreps('2x1o + 2x0e')

# node_attr_irreps=Irreps.spherical_harmonics(lmax=args.node_lmax)
node_attr_irreps = Irreps('0e+1o')


# black box to specify the hidden layer, with some parametrization
hidden_irreps = BalancedIrreps(lmax=args.hidden_lmax, vec_dim=args.hidden_size)

# y: node features to predict [P,vx,vy,vz]
output_irreps = Irreps('0e + 1o')

# edge features (relative distance vector and norm)
edge_attr_irreps = Irreps.spherical_harmonics(lmax=args.edge_lmax)

# no additional attributes specified
additional_message_irreps = None # Irreps('1x0e')

model = SEGNN(hidden_irreps=hidden_irreps,
              output_irreps=output_irreps,
              edge_attr_irreps=edge_attr_irreps,
              node_attr_irreps=node_attr_irreps,
              input_irreps=input_irreps,
              task=args.task,
              norm=args.norm,
              num_layers=args.num_layers,
              additional_message_irreps=additional_message_irreps
              )

model.to(dev)

SEGNN(
  (embedding_layer): O3TensorProduct(
    (tp): FullyConnectedTensorProduct(2x1o+2x0e x 1x0e+1x1o -> 130x0e+42x1o | 688 paths | 688 weights)
  )
  (layers): ModuleList(
    (0): SEGNNLayer()
    (1): SEGNNLayer()
    (2): SEGNNLayer()
    (3): SEGNNLayer()
    (4): SEGNNLayer()
    (5): SEGNNLayer()
    (6): SEGNNLayer()
    (7): SEGNNLayer()
  )
  (pre_pool1): O3TensorProductSwishGate(
    (tp): FullyConnectedTensorProduct(130x0e+42x1o x 1x0e+1x1o -> 172x0e+42x1o | 36808 paths | 36808 weights)
    (gate): Gate (172x0e+42x1o -> 130x0e+42x1o)
  )
  (pre_pool2): O3TensorProduct(
    (tp): FullyConnectedTensorProduct(130x0e+42x1o x 1x0e+1x1o -> 1x0e+1x1o | 344 paths | 344 weights)
  )
)

In [13]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
nparams = count_parameters(model)
print('Number of parameters; ',nparams)

mem_params = sum([param.nelement()*param.element_size() for param in model.parameters()])
mem_bufs = sum([buf.nelement()*buf.element_size() for buf in model.buffers()])
print()
print('Memory (MB) occupied by parameters and buffers: ',mem_bufs, '\t', mem_params)
print('Memory allocated on GPU:', torch.cuda.memory_allocated()*4/(1024**2), "MB")
print('Peak memory allocated on GPU:', torch.cuda.max_memory_allocated()*4/(1024**2), "MB")
# print(torch.cuda.memory_summary())

Number of parameters;  1757135

Memory (MB) occupied by parameters and buffers:  153760 	 7028540
Memory allocated on GPU: 0.0 MB
Peak memory allocated on GPU: 0.0 MB


In [14]:
!nvidia-smi

"nvidia-smi" non � riconosciuto come comando interno o esterno,
 un programma eseguibile o un file batch.


# **Loss and optimizer**

In [15]:
loss_func = nn.MSELoss()
if args.opt == 'Adam':
    opt = optim.Adam(model.parameters(), lr = args.learning_rate)
if args.scheduler == 'ExponentialLR':
    scheduler = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=0.9261) # 0.8317 25 epochs, 0.8913 40 epochs 0.9261 60 epoch
else:
    scheduler = None    

model_name = name + '_lat'+str(args.hidden_size) + '_knn' + \
            str(args.neighbours) + '_bs' + str(args.batch_size) + '_lr' + str(args.learning_rate) + '_ep' + \
            str(args.epochs) + '_subset' + str(args.subsample_dataset) + '_nparams' + str(nparams) +'.pt'

In [16]:
instance_norm = True

input_norm = InstanceNorm(input_irreps)
edge_norm = InstanceNorm(edge_attr_irreps)
# node_norm = InstanceNorm(node_attr_irreps)
# output_norm = InstanceNorm(output_irreps)

In [17]:
def train(model, loader, opt, loss_func, dev, log = True, mask = False):
     
    model.train()

    training_loss = []       
    for sample in loader:
        
        #print('SM...sample.x', sample.x.shape)
        edge_index = knn_graph(sample.pos, args.neighbours, sample.batch)
        sample.edge_index = edge_index
        
        # computes relative positions for every node pair
        edge_relativePos = (torch.index_select(sample.pos, 0, edge_index[1]) - torch.index_select(sample.pos, 0, edge_index[0]))
        # computes distances between pairs
        edge_relativeDist = torch.norm(edge_relativePos, dim = -1, keepdim = True) 
        # concatenates relative pos and distance for every edge
        edge_attr = torch.cat([edge_relativeDist, edge_relativePos], dim = -1) 
        
        sample.edge_attr = edge_attr
        # sample.node_attr = sample.pos
        
        if instance_norm:
            sample.x = input_norm(sample.x, sample.batch)
            sample.edge_attr = edge_norm(sample.edge_attr, sample.edge_index[1,:])

        sample = sample.to(dev)
        
        fluid_nodes = torch.tensor(2)
        
        if mask:
            # sample.mask: node next to inlet
            # take mask of nodes far from inlet
            loss_mask = ~sample.mask
        else:
            loss_mask = torch.ones(sample.x.shape[0], dtype=torch.bool)


        opt.zero_grad()
    
        pred = model(sample)      
        # print('train', pred.shape, pred[loss_mask].shape, sample.y.shape)
        loss = loss_func(pred[loss_mask], sample.y[loss_mask])  

        loss.backward()
        opt.step() 
        
        training_loss.append(loss)
        

        # if log == True:
        #     wandb.log({"train/batch_train_loss": loss})

        
    return sum(training_loss) / len(loader)



def val(model, loader, loss_func, mask = False):
    
    model.eval()
    
    with torch.no_grad():
   
        validation_loss = [] 
        for sample in loader:
            
            edge_index = knn_graph(sample.pos, args.neighbours, sample.batch)
            sample.edge_index = edge_index
            
            edge_relativePos = (torch.index_select(sample.pos, 0, edge_index[1]) - torch.index_select(sample.pos, 0, edge_index[0]))
            edge_relativeDist = torch.norm(edge_relativePos, dim = -1, keepdim = True) 
            edge_attr = torch.cat([edge_relativeDist, edge_relativePos], dim = -1) 
            
            sample.edge_attr = edge_attr
            # sample.node_attr = sample.pos
            
            if instance_norm:
                sample.x = input_norm(sample.x, sample.batch)
                sample.edge_attr = edge_norm(sample.edge_attr, sample.edge_index[1,:])
            
            sample = sample.to(dev)
            
            fluid_nodes = torch.tensor(2)
            if mask:
                # sample.mask: node next to inlet
                # take mask of nodes far from inlet
                loss_mask = ~sample.mask
            else:
                loss_mask = torch.ones(sample.x.shape[0], dtype=torch.bool)          

            pred = model(sample)
            # print('validation', pred.shape, pred[loss_mask].shape, sample.y.shape)
            loss = loss_func(pred[loss_mask], sample.y[loss_mask])  
            validation_loss.append(loss)

    return sum(validation_loss) / len(loader)


def early_stopping(val_loss, best_loss, counter):
        
    if val_loss < best_loss:
        best_loss = val_loss
        counter = 0
    else:
        counter += 1
    return counter, best_loss

# **Training and validation**

In [18]:
samples = 0
best_loss = 10**6
counter = 0
log = True

# wandb.init(  
#       project= project,    
#       config={
#         "epochs": args.epochs,
#         "bs": args.batch_size,
#         "lr": args.learning_rate,
#         "neighbours": args.neighbours,
#         "latent size": args.hidden_size,
#         "model parameters": nparams
#         })

path = os.path.join(params.DATADIR, 'checkpoints', model_name)

#wandb.run.name = model_name

mask = True
# mask = False

for epoch in range(args.epochs):

    # training
    train_loss = train(model, t_loader, opt, loss_func, dev, log, mask)
    if scheduler != None: 
        scheduler.step()
    samples += len(train_dataset)

    metrics = {"train/train_loss": train_loss,  
                "train/samples": samples}
    #wandb.log(metrics)    
    
    # validation
    val_loss = val(model, v_loader, loss_func, mask)

    val_metrics = {"val/val_loss": val_loss,
                   "epoch": epoch+1}

    #wandb.log(val_metrics)
    
    # early stopping
    counter, best_loss = early_stopping(val_loss, best_loss, counter)
    if counter == 0:

        checkpoint = {
            'epoch': epoch+1,
            'model': model,
            'optimizer_state_dict': opt.state_dict(),
            'training_loss': train_loss,
            'validation_loss': val_loss,
            'input_irreps':input_irreps,
            'hidden_irreps': hidden_irreps,
            'output_irreps': output_irreps,
            'edge_attr_irreps': edge_attr_irreps,
            'node_attr_irreps': node_attr_irreps,
            'task': args.task,
            'norm': args.norm,
            'num_layers': args.num_layers,
            'additional_message_irreps': additional_message_irreps
        }


    # if counter >= args.early_stop:        

    #     torch.save(checkpoint, path)
    #     # wandb.alert(
    #     #     title="Early stopping on validation data", 
    #     #     text=f"Loss {best_loss} was the best result, now is overfitting")
    #     # break

    
    print("Epoch " + str(epoch+1) + ": T loss " + str(train_loss))# + " V loss " + str(val_loss))
        
torch.save(checkpoint, path)
#wandb.finish()

c:\Users\crist\anaconda3\envs\segnn\lib\site-packages\torch_geometric\deprecation.py:13: UserWarning: 'contains_isolated_nodes' is deprecated, use 'has_isolated_nodes' instead
  warnings.warn(out)


Epoch 1: T loss tensor(0.4468, grad_fn=<DivBackward0>)
Epoch 2: T loss tensor(0.0116, grad_fn=<DivBackward0>)
Epoch 3: T loss tensor(0.0077, grad_fn=<DivBackward0>)
